In [2]:
import os
import gc
import copy
import glob
import pickle
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

H, SEQ = 28, 56

START_DAY, END_DAY = 1, 1913
TRAIN_TARGET_START, TRAIN_TARGET_END = 57, 1857
VAL_START, VAL_END = 1858, 1885
TEST_START, TEST_END = 1886, 1913

PRICE_STATS_END_DAY = TRAIN_TARGET_END
MEDIAN_FILL_END_DAY = TRAIN_TARGET_END

PREPROCESS_VERSION = "v6_matrix_panel_mse_d1_d1913"
CONFIG_VERSION = "boundaries_d57_1857_val1858_1885_test1886_1913"

_boundary_errors = []
if H != 28 or SEQ != 56: _boundary_errors.append("SEQ/H must be 56/28")
if TRAIN_TARGET_START != SEQ + 1: _boundary_errors.append("TRAIN_TARGET_START must equal SEQ+1")
if VAL_START != TRAIN_TARGET_END + 1: _boundary_errors.append("VAL_START must equal TRAIN_TARGET_END+1")
if VAL_END - VAL_START + 1 != H: _boundary_errors.append("VAL window must have H days")
if TEST_START != VAL_END + 1: _boundary_errors.append("TEST_START must equal VAL_END+1")
if TEST_END - TEST_START + 1 != H: _boundary_errors.append("TEST window must have H days")
if TEST_END != END_DAY: _boundary_errors.append("TEST_END must equal END_DAY")
if PRICE_STATS_END_DAY > TRAIN_TARGET_END or MEDIAN_FILL_END_DAY > TRAIN_TARGET_END:
    _boundary_errors.append("preprocessing cutoff after train end (leak)")

if _boundary_errors:
    raise ValueError("Invalid boundary config: " + " | ".join(_boundary_errors))
del _boundary_errors

TRAIN_SAMPLE_N = None
LOG_TARGET = True

TFT_CFG = {
    "d_model": 64,
    "num_heads": 4,
    "epochs": 18,
    "batch_size": 512,
    "lr": 1e-3,
    "dropout": 0.1,
    "patience": 5,
}

SAMPLING = {
    "recent_days": 300,
    "mid_days": 700,
    "recent_stride": 4,
    "mid_stride": 12,
    "old_stride": 48,
}

TFT_DATA_WORKERS = 0
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

torch: 2.11.0+cpu | cuda: False
DEVICE: cpu


In [3]:
DATA_ROOT = "/kaggle/input"
required_files = ["calendar.csv", "sales_train_validation.csv", "sales_train_evaluation.csv", "sell_prices.csv"]
DATA_DIR = None

calendar_matches = glob.glob(os.path.join(DATA_ROOT, "**", "calendar.csv"), recursive=True)
for calendar_path in calendar_matches:
    candidate_dir = os.path.dirname(calendar_path)
    if all(os.path.isfile(os.path.join(candidate_dir, filename)) for filename in required_files):
        DATA_DIR = candidate_dir
        break

if DATA_DIR is None:
    raise FileNotFoundError("Cannot locate the complete M5 dataset.")

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)

DATA_DIR: /kaggle/input/datasets/vietnguyenthe123/m5-dataset
OUT_DIR: /kaggle/working


In [4]:
def _as_float(a):
    return np.asarray(a, dtype=np.float64).ravel()

def calc_rmse(a, b):
    a, b = _as_float(a), _as_float(b)
    return float(np.sqrt(np.mean((a - b) ** 2))) if a.size else float('nan')

def calc_mae(a, b):
    a, b = _as_float(a), _as_float(b)
    return float(np.mean(np.abs(a - b))) if a.size else float('nan')

def calc_mape(y_true, y_pred):
    a, p = _as_float(y_true), _as_float(y_pred)
    mask = (a != 0) & ~np.isnan(a)
    if not np.any(mask): return float('nan')
    return float(np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100.0)

def calc_wmape(a, b):
    a, b = _as_float(a), _as_float(b)
    den = np.abs(a).sum()
    return float('nan') if den == 0 else float(np.abs(a - b).sum() / den * 100)

def calc_bias(a, b):
    a, b = _as_float(a), _as_float(b)
    return float(np.mean(b - a)) if a.size else float('nan')

def calc_smape(a, b, eps=1e-8):
    a, b = _as_float(a), _as_float(b)
    den = (np.abs(a) + np.abs(b)) / 2.0 + eps
    return float(np.mean(np.abs(b - a) / den) * 100) if a.size else float('nan')

def calc_rmsse(y_true, y_pred, y_train):
    a, p, tr = _as_float(y_true), _as_float(y_pred), _as_float(y_train)
    nz = np.where(tr != 0)[0]
    if len(nz) == 0:
        scale_sq = 0.0
    else:
        tr_act = tr[nz[0]:]
        scale_sq = np.mean(np.diff(tr_act) ** 2) if len(tr_act) > 1 else 0.0
    rmse = float(np.sqrt(np.mean((a - p) ** 2))) if a.size else float('nan')
    if scale_sq <= 0 or np.isnan(scale_sq):
        return 0.0 if rmse == 0 else float('inf')
    return float(rmse / np.sqrt(scale_sq))

def forecast_metrics(y_true, y_pred, y_train=None, label=''):
    a, p = _as_float(y_true), _as_float(y_pred)
    den = np.abs(a).sum()
    ratio = float(p.sum() / a.sum()) if a.sum() != 0 else float('nan')
    res = {
        'n_obs': int(a.size), 'total_actual': float(a.sum()), 'total_pred': float(p.sum()),
        'pred_actual_ratio': ratio, 'WMAPE (%)': calc_wmape(a, p), 'MAPE (%)': calc_mape(a, p),
        'MAE': calc_mae(a, p), 'RMSE': calc_rmse(a, p), 'Bias': calc_bias(a, p),
        'sMAPE (%)': calc_smape(a, p), 'zero_actual_frac': float(np.mean(a == 0)) if a.size else float('nan'),
        'zero_denominator': bool(den == 0),
    }
    if y_train is not None: res['RMSSE'] = calc_rmsse(a, p, y_train)
    return res

print('utils OK')

utils OK


In [5]:
import gc, time, warnings
import numpy as np
import pandas as pd
from scipy import sparse

T_PIPE = time.time()
N_DAYS = END_DAY - START_DAY + 1

cal = pd.read_csv(os.path.join(DATA_DIR, 'calendar.csv'))
cal['d_num'] = cal['d'].str[2:].astype(np.int32)
cal = cal[(cal['d_num'] >= START_DAY) & (cal['d_num'] <= END_DAY)].sort_values('d_num').reset_index(drop=True)

_wday = cal['wday'].to_numpy(np.int32)
_month = cal['month'].to_numpy(np.int32)
CAL_WM_YR_WK = cal['wm_yr_wk'].to_numpy(np.int32)
CAL_IS_WEEKEND = cal['wday'].isin([1, 2]).to_numpy(np.int8)
CAL_SIN_WDAY = np.sin(2 * np.pi * _wday / 7.0).astype(np.float32)
CAL_COS_WDAY = np.cos(2 * np.pi * _wday / 7.0).astype(np.float32)
CAL_SIN_MONTH = np.sin(2 * np.pi * _month / 12.0).astype(np.float32)
CAL_COS_MONTH = np.cos(2 * np.pi * _month / 12.0).astype(np.float32)

for _c in ['event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']:
    if _c not in cal.columns: cal[_c] = 'None'
    cal[_c] = cal[_c].fillna('None')
CAL_HAS_EVENT = (cal['event_name_1'] != 'None').to_numpy(np.int8)
CAL_EVENT_COUNT = ((cal['event_name_1'] != 'None').to_numpy(np.int32) + (cal['event_name_2'] != 'None').to_numpy(np.int32)).astype(np.int8)
CAL_EVT = {_t: (cal['event_type_1'] == _t).to_numpy(np.int8) for _t in ['Cultural', 'Religious', 'Sporting', 'National']}

_n = len(cal)
_evt_idx = np.where(CAL_HAS_EVENT == 1)[0]
_d2n = np.full(_n, 999, dtype=np.int16)
_dsl = np.full(_n, 999, dtype=np.int16)
if len(_evt_idx) > 0:
    _nx = np.searchsorted(_evt_idx, np.arange(_n))
    _v = _nx < len(_evt_idx)
    _d2n[_v] = (_evt_idx[_nx[_v]] - np.arange(_n)[_v]).astype(np.int16)
    _pv = np.searchsorted(_evt_idx, np.arange(_n), side='right') - 1
    _vp = _pv >= 0
    _dsl[_vp] = (np.arange(_n)[_vp] - _evt_idx[_pv[_vp]]).astype(np.int16)
CAL_DAYS_TO_NEXT = np.clip(_d2n, 0, 60).astype(np.int16)
CAL_DAYS_SINCE = np.clip(_dsl, 0, 60).astype(np.int16)

CAL_SNAP = {}
for _st in ['CA', 'TX', 'WI']:
    _c = f'snap_{_st}'
    CAL_SNAP[_st] = cal[_c].to_numpy(np.int8) if _c in cal.columns else np.zeros(_n, np.int8)
del cal; gc.collect()

_meta_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
_sales_path = os.path.join(DATA_DIR, 'sales_train_validation.csv')
_day_cols = [f'd_{i}' for i in range(START_DAY, END_DAY + 1)]
_sales = pd.read_csv(_sales_path, usecols=_meta_cols + _day_cols, dtype={c: np.int16 for c in _day_cols})
SALES = np.ascontiguousarray(_sales[_day_cols].to_numpy(np.int16))
N_SERIES = SALES.shape[0]
SERIES_ID = _sales['id'].astype(str).to_numpy()
META_ITEM = _sales['item_id'].astype(str).to_numpy()
META_DEPT = _sales['dept_id'].astype(str).to_numpy()
META_CAT = _sales['cat_id'].astype(str).to_numpy()
META_STORE = _sales['store_id'].astype(str).to_numpy()
META_STATE = _sales['state_id'].astype(str).to_numpy()
del _sales; gc.collect()

def _codes(arr):
    uniq = np.array(sorted(set(arr.tolist())))
    lut = {v: i + 1 for i, v in enumerate(uniq)}
    return uniq, np.array([lut[v] for v in arr], dtype=np.int64)

STORE_NAMES, STORE_CODES = _codes(META_STORE)
STATE_NAMES, STATE_CODES = _codes(META_STATE)
CAT_NAMES, CAT_CODES = _codes(META_CAT)
DEPT_NAMES, DEPT_CODES = _codes(META_DEPT)
ITEM_NAMES, ITEM_CODES = _codes(META_ITEM)
N_STORES, N_STATES, N_CATS, N_DEPTS, N_ITEMS = (len(STORE_NAMES), len(STATE_NAMES), len(CAT_NAMES), len(DEPT_NAMES), len(ITEM_NAMES))

_weeks = np.unique(CAL_WM_YR_WK)
WEEKS = _weeks.astype(np.int32)
N_WEEKS = len(WEEKS)
_wk_col = np.searchsorted(WEEKS, CAL_WM_YR_WK)
_wk_train_end = int(CAL_WM_YR_WK[PRICE_STATS_END_DAY - START_DAY])
_week_is_train = WEEKS <= _wk_train_end

_prices = pd.read_csv(os.path.join(DATA_DIR, 'sell_prices.csv'), dtype={'sell_price': np.float32})
_price_wk = _prices['wm_yr_wk'].to_numpy(np.int32)
_keep = np.isin(_price_wk, WEEKS)
_price_wk = _price_wk[_keep]
_price_val = _prices['sell_price'].to_numpy(np.float32)[_keep]
_price_store = pd.Categorical(_prices['store_id'].astype(str), categories=list(STORE_NAMES)).codes[_keep]
_price_item = pd.Categorical(_prices['item_id'].astype(str), categories=list(ITEM_NAMES)).codes[_keep]
del _prices; gc.collect()

_price_wc = np.searchsorted(WEEKS, _price_wk)
_pair = np.full((N_ITEMS, N_STORES), -1, dtype=np.int64)
_pair[ITEM_CODES - 1, STORE_CODES - 1] = np.arange(N_SERIES)
_price_sidx = _pair[_price_item, _price_store]
_ok = _price_sidx >= 0
PRICE_WEEK = np.full((N_SERIES, N_WEEKS), np.nan, dtype=np.float32)
PRICE_WEEK[_price_sidx[_ok], _price_wc[_ok]] = _price_val[_ok]
del _price_sidx, _price_val, _price_wk, _price_wc, _price_store, _price_item, _keep, _ok, _pair; gc.collect()

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _ptrain = PRICE_WEEK[:, _week_is_train]
    PRICE_MAX = np.nanmax(_ptrain, axis=1).astype(np.float32)
    del _ptrain

with np.errstate(divide='ignore', invalid='ignore'):
    _disc_week = np.where(np.isnan(PRICE_WEEK), np.nan, 1.0 - PRICE_WEEK / PRICE_MAX[:, None])
_disc_week = np.nan_to_num(_disc_week, nan=0.0, posinf=0.0, neginf=0.0)
np.clip(_disc_week, 0.0, 1.0, out=_disc_week)

_present = ~np.isnan(PRICE_WEEK)
_idx_present = np.where(_present, np.arange(N_WEEKS)[None, :], -1)
_prev = np.maximum.accumulate(_idx_present, axis=1)
_prev = np.roll(_prev, 1, axis=1)
_prev[:, 0] = -1
_prev_price = np.where(_prev >= 0, PRICE_WEEK[np.arange(N_SERIES)[:, None], np.clip(_prev, 0, None)], np.nan)
with np.errstate(divide='ignore', invalid='ignore'):
    _mom_week = PRICE_WEEK / _prev_price - 1.0
_mom_week = np.nan_to_num(_mom_week, nan=0.0, posinf=0.0, neginf=0.0)
del _present, _idx_present, _prev, _prev_price; gc.collect()

PRICE_DAY = np.ascontiguousarray(PRICE_WEEK[:, _wk_col])
DISC_DAY = np.ascontiguousarray(_disc_week[:, _wk_col])
MOM_DAY = np.ascontiguousarray(_mom_week[:, _wk_col])
del PRICE_WEEK, _disc_week, _mom_week; gc.collect()

_n_train_days = MEDIAN_FILL_END_DAY - START_DAY + 1
PRICE_FILLED = PRICE_DAY.copy()
IS_LISTED = ~np.isnan(PRICE_DAY)
_dept_med = np.full(N_SERIES, np.nan, dtype=np.float64)
for _d in range(1, N_DEPTS + 1):
    _rows = np.where(DEPT_CODES == _d)[0]
    if len(_rows) == 0: continue
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        _dept_med[_rows] = np.nanmedian(PRICE_DAY[_rows, :_n_train_days], axis=1)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    _glob_med = float(np.nanmedian(PRICE_DAY[:, :_n_train_days]))
_nanmask = np.isnan(PRICE_FILLED)
if _nanmask.any():
    _rows, _cols = np.nonzero(_nanmask)
    _fill_vals = _dept_med[_rows]
    _bad = np.isnan(_fill_vals)
    _fill_vals[_bad] = _glob_med
    PRICE_FILLED[_rows, _cols] = _fill_vals
    del _rows, _cols, _fill_vals
del _nanmask; gc.collect()

PAST_NAMES = [
    'sales', 'sales_lag_28', 'sales_lag_35', 'sales_rolling_mean_7', 'sales_rolling_mean_28',
    'sell_price_lag_28', 'price_discount_lag_28', 'price_momentum_lag_28',
    'sell_price_rolling_mean_7', 'is_listed', 'is_listed_lag_28',
    'days_since_last_sale', 'nonzero_ratio_28', 'nonzero_ratio_56', 'sales_rolling_mean_56',
]
FUT_NAMES = [
    'is_weekend', 'sin_wday', 'cos_wday', 'sin_month', 'cos_month', 'snap_active',
    'has_event', 'event_count', 'evt_Cultural', 'evt_Religious', 'evt_Sporting',
    'evt_National', 'days_to_next_event', 'days_since_last_event',
]
STATIC_NAMES = ['store_id', 'state_id', 'cat_id', 'dept_id', 'item_id']

PAST = np.zeros((len(PAST_NAMES), N_SERIES, N_DAYS), dtype=np.float32)

def _shift_into(dst, src, k):
    dst[:, k:] = src[:, :-k]

def _roll_mean_shift_into(dst, src, w, shift=28):
    S, D = src.shape
    chunk = 4000
    for lo in range(0, S, chunk):
        hi = min(S, lo + chunk)
        cs = np.zeros((hi - lo, D + 1), dtype=np.float64)
        np.cumsum(src[lo:hi], axis=1, dtype=np.float64, out=cs[:, 1:])
        t = np.arange(D)
        a = t - shift
        b = np.maximum(a - w + 1, 0)
        valid = a >= 0
        a_c = np.clip(a, 0, D)
        b_c = np.clip(b, 0, D)
        sums = cs[:, a_c + 1] - cs[:, b_c]
        counts = np.where(valid, a - b + 1, 0)
        out = dst[lo:hi]
        np.divide(sums, np.maximum(counts, 1)[None, :], out=out, where=valid[None, :])
        out[:, ~valid] = 0.0
        del cs, sums
    return dst

PAST[0] = SALES
_shift_into(PAST[1], SALES, 28)
_shift_into(PAST[2], SALES, 35)
_roll_mean_shift_into(PAST[3], SALES, 7, 28)
_roll_mean_shift_into(PAST[4], SALES, 28, 28)
_shift_into(PAST[5], PRICE_FILLED, 28)
_shift_into(PAST[6], DISC_DAY, 28)
_shift_into(PAST[7], MOM_DAY, 28)
_roll_mean_shift_into(PAST[8], PRICE_FILLED, 7, 28)
PAST[9] = IS_LISTED
_shift_into(PAST[10], IS_LISTED, 28)

_t_arr = np.arange(N_DAYS, dtype=np.int32)
_nz = (PAST[0] > 0)
for _lo in range(0, N_SERIES, 4000):
    _hi = min(N_SERIES, _lo + 4000)
    _idx = np.where(_nz[_lo:_hi], _t_arr[None, :], np.int32(-1))
    _last = np.maximum.accumulate(_idx, axis=1)
    _dsls = np.where(_last >= 0, _t_arr[None, :] - _last, _t_arr[None, :] + 1)
    PAST[11, _lo:_hi] = np.clip(_dsls, 0, 28).astype(np.float32)
    del _idx, _last, _dsls
_roll_mean_shift_into(PAST[12], _nz, 28, 0)
_roll_mean_shift_into(PAST[13], _nz, 56, 0)
_roll_mean_shift_into(PAST[14], PAST[0], 56, 0)
del _nz, _t_arr

PRICE_MATRIX = PRICE_DAY.copy()
del PRICE_FILLED, PRICE_DAY, DISC_DAY, MOM_DAY, IS_LISTED, SALES; gc.collect()

_active = (PAST[0] > 0) | (PAST[9] > 0)
_has_any = _active.any(axis=1)
START_IDX = np.argmax(_active, axis=1).astype(np.int32)
START_IDX[~_has_any] = -1
del _active; gc.collect()

_day_feats = np.stack([
    CAL_IS_WEEKEND, CAL_SIN_WDAY, CAL_COS_WDAY, CAL_SIN_MONTH, CAL_COS_MONTH,
    CAL_HAS_EVENT, CAL_EVENT_COUNT, CAL_EVT['Cultural'], CAL_EVT['Religious'],
    CAL_EVT['Sporting'], CAL_EVT['National'], CAL_DAYS_TO_NEXT, CAL_DAYS_SINCE,
], axis=1).astype(np.float32)

_store_state = {}
for _i in range(N_STORES):
    _st = np.unique(META_STATE[STORE_CODES == (_i + 1)])
    _store_state[_i] = str(_st[0])

FUT = np.empty((N_STORES, N_DAYS, len(FUT_NAMES)), dtype=np.float32)
for _i in range(N_STORES):
    _snap = CAL_SNAP[_store_state[_i]].astype(np.float32)
    FUT[_i, :, :5] = _day_feats[:, 0:5]
    FUT[_i, :, 5] = _snap
    FUT[_i, :, 6:] = _day_feats[:, 5:]
del _day_feats; gc.collect()

STATIC = np.zeros((N_SERIES, len(STATIC_NAMES)), dtype=np.int64)
STATIC[:, 0] = STORE_CODES; STATIC[:, 1] = STATE_CODES; STATIC[:, 2] = CAT_CODES; STATIC[:, 3] = DEPT_CODES; STATIC[:, 4] = ITEM_CODES
STATIC_VALUES = {'store_id': STORE_NAMES, 'state_id': STATE_NAMES, 'cat_id': CAT_NAMES, 'dept_id': DEPT_NAMES, 'item_id': ITEM_NAMES}
SERIES_STORE_IDX = STORE_CODES - 1

def _group_matrix(codes, n_keys):
    rows = np.asarray(codes, dtype=np.int64) - 1
    cols = np.arange(len(rows), dtype=np.int64)
    return sparse.csr_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(n_keys, len(rows)))

_SD_CODE = (STORE_CODES * 1000 + DEPT_CODES)
_SD_UNIQ, _SD_INV = np.unique(_SD_CODE, return_inverse=True)
L9_KEYS = [f'{STORE_NAMES[s - 1]}|{DEPT_NAMES[d - 1]}' for (s, d) in sorted({(int(STORE_CODES[i]), int(DEPT_CODES[i])) for i in range(N_SERIES)})]

GROUPS = {
    'L1': (['ALL'], sparse.csr_matrix(np.ones((1, N_SERIES), np.float32))),
    'L2': (list(STATE_NAMES), _group_matrix(STATE_CODES, N_STATES)),
    'L3': (list(STORE_NAMES), _group_matrix(STORE_CODES, N_STORES)),
    'L9': (L9_KEYS, _group_matrix(_SD_INV + 1, len(_SD_UNIQ))),
    'L10': (list(ITEM_NAMES), _group_matrix(ITEM_CODES, N_ITEMS)),
}
del _SD_CODE, _SD_UNIQ, _SD_INV; gc.collect()

class M5Panel:
    def __init__(self, past, past_names, fut, fut_names, static, static_names,
                 static_values, store_idx, start_idx, ids, item, dept, cat, store, state, groups, prices=None):
        self.past = past; self.past_names = list(past_names)
        self.fut = fut; self.fut_names = list(fut_names)
        self.static = static; self.static_names = list(static_names)
        self.static_values = static_values; self.store_idx = store_idx
        self.start_idx = start_idx; self.ids = ids
        self.item = item; self.dept = dept; self.cat = cat; self.store = store; self.state = state
        self.groups = groups; self.prices = prices
        self.n_series = past.shape[1]; self.n_days = past.shape[2]

    def memory_mb(self):
        return (self.past.nbytes + self.fut.nbytes + self.static.nbytes) / 1024 ** 2

PANEL = M5Panel(PAST, PAST_NAMES, FUT, FUT_NAMES, STATIC, STATIC_NAMES, STATIC_VALUES,
                SERIES_STORE_IDX, START_IDX, SERIES_ID, META_ITEM, META_DEPT, META_CAT,
                META_STORE, META_STATE, GROUPS, prices=PRICE_MATRIX)
del PAST, FUT, STATIC, SERIES_STORE_IDX, START_IDX, SERIES_ID, META_ITEM, META_DEPT, META_CAT, META_STORE, META_STATE, GROUPS, PRICE_MATRIX; gc.collect()

print(f'PANEL RAM: {PANEL.memory_mb():,.0f} MB | series={PANEL.n_series:,} days={PANEL.n_days:,}')

PANEL RAM: 3,340 MB | series=30,490 days=1,913


In [6]:
import copy, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

def _new_amp():
    try:
        return torch.amp.GradScaler('cuda'), True
    except Exception:
        return torch.cuda.amp.GradScaler(), False

def _autocast():
    try:
        return torch.amp.autocast('cuda')
    except Exception:
        return torch.cuda.amp.autocast()

class GLU(nn.Module):
    def __init__(self, inp, out):
        super().__init__()
        self.fc_val = nn.Linear(inp, out); self.fc_gate = nn.Linear(inp, out)
    def forward(self, x):
        return self.fc_val(x) * torch.sigmoid(self.fc_gate(x))

class GRN(nn.Module):
    def __init__(self, inp, hid, out, dropout=0.1, context_dim=None):
        super().__init__()
        self.fc1 = nn.Linear(inp, hid)
        self.fc_context = nn.Linear(context_dim, hid, bias=False) if context_dim else None
        self.fc2 = nn.Linear(hid, hid); self.drop = nn.Dropout(dropout)
        self.glu = GLU(hid, out); self.norm = nn.LayerNorm(out)
        self.skip = nn.Linear(inp, out) if inp != out else nn.Identity()
    def forward(self, x, context=None):
        h = self.fc1(x)
        if context is not None and self.fc_context is not None:
            ctx = context.unsqueeze(1) if h.dim() == 3 else context
            h = h + self.fc_context(ctx)
        h = self.drop(self.fc2(F.elu(h)))
        return self.norm(self.skip(x) + self.glu(h))

class VSN(nn.Module):
    def __init__(self, n_vars, inp, hid, dropout=0.1, context_dim=None):
        super().__init__()
        self.var_grns = nn.ModuleList([GRN(inp, hid, hid, dropout) for _ in range(n_vars)])
        self.weight_grn = GRN(n_vars * inp, hid, n_vars, dropout, context_dim)
    def forward(self, flat, indiv, context=None):
        w = F.softmax(self.weight_grn(flat, context), dim=-1)
        stacked = torch.stack([g(v) for g, v in zip(self.var_grns, indiv)], dim=-2)
        return torch.sum(stacked * w.unsqueeze(-1), dim=-2), w

class InterpretableMHA(nn.Module):
    def __init__(self, heads, d_model, dropout=0.1):
        super().__init__()
        assert d_model % heads == 0
        self.h = heads; self.dk = d_model // heads
        self.q = nn.Linear(d_model, d_model, bias=False)
        self.k = nn.Linear(d_model, d_model, bias=False)
        self.v = nn.Linear(d_model, d_model, bias=False)
        self.o = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
    def forward(self, q, k, v, mask=None):
        B = q.size(0)
        qh = self.q(q).view(B, -1, self.h, self.dk).transpose(1, 2)
        kh = self.k(k).view(B, -1, self.h, self.dk).transpose(1, 2)
        vs = self.v(v)
        sc = torch.matmul(qh, kh.transpose(-2, -1)) / (self.dk ** 0.5)
        if mask is not None:
            sc = sc.masked_fill(mask == 0, torch.finfo(sc.dtype).min)
        aw = self.drop(F.softmax(sc, dim=-1))
        mean_aw = aw.mean(dim=1)
        return self.o(torch.matmul(mean_aw, vs)), mean_aw

class StaticContext(nn.Module):
    def __init__(self, cards, d):
        super().__init__()
        self.embs = nn.ModuleDict({k: nn.Embedding(c + 1, d, padding_idx=0) for k, c in cards.items()})
    def forward(self, sx):
        return sum(self.embs[k](sx[k]) for k in self.embs)

class TFTGlobalNet(nn.Module):
    def __init__(self, npast, nfut, cards, d=16, heads=2, drop=0.1, seq=56, H=28, qs=[0.1, 0.5, 0.9]):
        super().__init__(); self.seq = seq
        self.ctx = StaticContext(cards, d)
        self.pe = nn.ModuleList([nn.Linear(1, d) for _ in range(npast)])
        self.fe = nn.ModuleList([nn.Linear(1, d) for _ in range(nfut)])
        self.pvsn = VSN(npast, d, d, drop, d); self.fvsn = VSN(nfut, d, d, drop, d)
        self.enc = nn.LSTM(d, d, batch_first=True); self.dec = nn.LSTM(d, d, batch_first=True)
        self.glu = GLU(d, d); self.norm = nn.LayerNorm(d)
        self.enrich = GRN(d, d, d, drop, d)
        self.attn = InterpretableMHA(heads, d, drop); self.post = GRN(d, d, d, drop)
        self.head = nn.Linear(d, len(qs)) # Vẫn giữ 3 outputs để không vỡ checkpoint
    def forward(self, px, fx, sx):
        se = self.ctx(sx)
        pe = [e(px[:, :, i:i+1]) for i, e in enumerate(self.pe)]
        pv, _ = self.pvsn(torch.cat(pe, -1), pe, se)
        fe = [e(fx[:, :, i:i+1]) for i, e in enumerate(self.fe)]
        fv, _ = self.fvsn(torch.cat(fe, -1), fe, se)
        enc, (hn, cn) = self.enc(pv)
        dec, _ = self.dec(fv[:, self.seq:], (hn, cn))
        comb = torch.cat([enc, dec], 1); vsn = torch.cat([pv, fv[:, self.seq:]], 1)
        tf = self.norm(vsn + self.glu(comb)); en = self.enrich(tf, se)
        L = en.size(1)
        mask = torch.tril(torch.ones(L, L, dtype=torch.bool, device=en.device))
        ao, _ = self.attn(en, en, en, mask)
        out = self.post(en + ao)
        return self.head(out[:, self.seq:]), {}

class MatrixTFTDataset(Dataset):
    def __init__(self, panel, eligible, seq=56, H=28, target_start=None, target_end=None,
                 p_mean=None, p_scale=None, f_mean=None, f_scale=None,
                 t_mean=0.0, t_scale=1.0, log_target=True,
                 recent_days=300, mid_days=700, recent_stride=4, mid_stride=12, old_stride=48):
        self.panel = panel; self.seq = seq; self.H = H
        self.p_mean = p_mean; self.p_scale = p_scale; self.f_mean = f_mean; self.f_scale = f_scale
        self.t_mean = t_mean; self.t_scale = t_scale; self.log_target = log_target
        if target_start is None: target_start = int(globals()['TRAIN_TARGET_START'])
        if target_end is None: target_end = int(globals()['TRAIN_TARGET_END'])
        self.target_start = int(target_start); self.target_end = int(target_end)
        starts, sids = [], []
        empty_series = []
        for s in eligible:
            d0 = int(panel.start_idx[s])
            if d0 < 0: empty_series.append(int(s)); continue
            n_avail = self.target_end - d0
            nw = max(0, n_avail - seq - H + 1)
            k, kept = 0, 0
            while k < nw:
                tgt_lo = d0 + k + seq + 1
                tgt_hi = tgt_lo + H - 1
                if tgt_lo < self.target_start or tgt_hi > self.target_end:
                    k += 1; continue
                age = self.target_end - tgt_hi
                st = recent_stride if age < recent_days else (mid_stride if age < recent_days + mid_days else old_stride)
                starts.append(d0 + k); sids.append(int(s)); kept += 1; k += st
            if kept == 0: empty_series.append(int(s))
        self.starts = np.asarray(starts, dtype=np.int64)
        self.sids = np.asarray(sids, dtype=np.int64)
        self.total = len(self.starts)
    def __len__(self): return self.total
    def __getitem__(self, idx): return (int(self.sids[idx]), int(self.starts[idx]))

def _extract_windows(panel, s_arr, a_arr, seq, H, p_mean, p_scale, f_mean, f_scale, t_mean, t_scale, log_target):
    s_arr, a_arr = np.asarray(s_arr, dtype=np.int64), np.asarray(a_arr, dtype=np.int64)
    widx = a_arr[:, None] + np.arange(seq + H, dtype=np.int64)[None, :]
    p = panel.past[:, s_arr[:, None], widx[:, :seq]].transpose(1, 2, 0)
    p = (p - p_mean[None, None, :]) / p_scale[None, None, :]
    f = panel.fut[panel.store_idx[s_arr][:, None], widx, :]
    f = (f - f_mean[None, None, :]) / f_scale[None, None, :]
    t = panel.past[0, s_arr[:, None], widx[:, seq:seq + H]]
    if log_target: t = np.log1p(t)
    t = (t - t_mean) / t_scale
    st = panel.static[s_arr]
    return p, f, t, st

def _make_collate(panel, seq, H, p_mean, p_scale, f_mean, f_scale, t_mean, t_scale, log_target):
    def collate(batch):
        s_arr = np.fromiter((b[0] for b in batch), dtype=np.int64, count=len(batch))
        a_arr = np.fromiter((b[1] for b in batch), dtype=np.int64, count=len(batch))
        p, f, t, st = _extract_windows(panel, s_arr, a_arr, seq, H, p_mean, p_scale, f_mean, f_scale, t_mean, t_scale, log_target)
        return (torch.from_numpy(p).float(), torch.from_numpy(f).float(), torch.from_numpy(st).long(), torch.from_numpy(t).float())
    return collate

class TFTGlobalForecaster:
    def __init__(self, past_cols, fut_cols, static_cols, target='sales', seq=56, H=28,
                 d=16, heads=2, lr=1e-3, batch=256, epochs=10, device=None,
                 log_target=True, dropout=0.1, patience=5, sampling=None):
        self.pc = past_cols; self.fc = fut_cols; self.sc = static_cols
        self.target = target; self.seq = seq; self.H = H
        self.d = d; self.heads = heads; self.lr = lr; self.batch = batch; self.epochs = epochs
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.log_target = log_target; self.dropout = dropout; self.patience = patience
        self.sampling = dict(sampling or {})
        self.ps = StandardScaler(); self.fs = StandardScaler(); self.ts = StandardScaler()
        self.maps = {}; self.cards = {}; self.model = None
        g = globals()
        self.train_target_start = int(g.get('TRAIN_TARGET_START', seq + 1))
        self.train_target_end = int(g.get('TRAIN_TARGET_END', g.get('END_DAY', 1913) - H))
        self.val_start = int(g.get('VAL_START', self.train_target_end + 1))
        self.val_end = int(g.get('VAL_END', self.val_start + H - 1))
        self.history = []; self.best_epoch = None; self.best_val_loss = None
        self.stop_reason = None; self.train_duration_s = None; self.peak_rss_mb = None; self.peak_gpu_mb = None
        self.n_windows = None; self.val_series = 0; self.val_skipped = 0
        self.p_mean = None; self.p_scale = None; self.f_mean = None; self.f_scale = None; self.t_mean = 0.0; self.t_scale = 1.0

    def _sync_scalers(self):
        self.p_mean = np.asarray(self.ps.mean_, dtype=np.float32)
        self.p_scale = np.asarray(self.ps.scale_, dtype=np.float32)
        self.f_mean = np.asarray(self.fs.mean_, dtype=np.float32)
        self.f_scale = np.asarray(self.fs.scale_, dtype=np.float32)
        self.t_mean = np.float32(self.ts.mean_[0]); self.t_scale = np.float32(self.ts.scale_[0])

    def _set_scaler(self, scaler, mean, scale, n):
        scaler.mean_ = np.asarray(mean, dtype=np.float64)
        scaler.var_ = np.asarray(scale, dtype=np.float64) ** 2
        scaler.scale_ = np.asarray(scale, dtype=np.float64)
        scaler.n_samples_seen_ = int(n); scaler.n_features_in_ = len(np.atleast_1d(mean))

    def _fit_scalers(self, panel, eligible):
        D = panel.n_days; train_end_idx = self.train_target_end - 1
        start = np.clip(panel.start_idx[eligible], 0, None).astype(np.int64)
        day = np.arange(D, dtype=np.int64)[None, :]
        valid = (day >= start[:, None]) & (day <= train_end_idx)
        cnt = int(valid.sum())
        def _stats(get_vals):
            s1 = 0.0; s2 = 0.0; n = 0
            for lo in range(0, len(eligible), 4000):
                sl = eligible[lo:lo + 4000]; m = valid[lo:lo + 4000]
                v = get_vals(sl); v = np.where(m, v, 0.0)
                s1 += float(v.sum(dtype=np.float64)); s2 += float((v * v).sum(dtype=np.float64))
                n += int(m.sum())
            mean = s1 / n; var = max(s2 / n - mean * mean, 0.0)
            scale = float(np.sqrt(var))
            if not np.isfinite(scale) or scale < 1e-8: scale = 1.0
            return mean, scale, n
        p_mean, p_scale = [], []
        for i in range(len(self.pc)):
            m, sc, _ = _stats(lambda sl, i=i: panel.past[i, sl, :].astype(np.float64))
            p_mean.append(m); p_scale.append(sc)
        self._set_scaler(self.ps, p_mean, p_scale, cnt)
        tm, ts_, _ = _stats(lambda sl: np.log1p(panel.past[0, sl, :].astype(np.float64)))
        self._set_scaler(self.ts, [tm], [ts_], cnt)

        f_sum, f_sq, w_tot = np.zeros(len(self.fc)), np.zeros(len(self.fc)), 0.0
        store_of = panel.store_idx[eligible]
        for st_i in range(panel.fut.shape[0]):
            rows = np.where(store_of == st_i)[0]
            if len(rows) == 0: continue
            starts = np.sort(panel.start_idx[eligible[rows]].astype(np.int64)); starts = starts[starts >= 0]
            if len(starts) == 0: continue
            w = np.searchsorted(starts, np.arange(D, dtype=np.int64), side='right').astype(np.float64)
            w[train_end_idx + 1:] = 0.0
            x = panel.fut[st_i, :train_end_idx + 1, :].astype(np.float64); ww = w[:train_end_idx + 1][:, None]
            f_sum += (x * ww).sum(axis=0); f_sq += (x * x * ww).sum(axis=0); w_tot += float(ww.sum())
        f_mean = f_sum / max(w_tot, 1.0); f_var = np.maximum(f_sq / max(w_tot, 1.0) - f_mean ** 2, 0.0)
        f_scale = np.sqrt(f_var); f_scale[~np.isfinite(f_scale) | (f_scale < 1e-8)] = 1.0
        self._set_scaler(self.fs, f_mean, f_scale, int(w_tot))
        self._sync_scalers()
        print(f'[scaler] past cells={cnt:,} | target mean={tm:.4f} scale={ts_:.4f} | fut weight={w_tot:,.0f}')

    @torch.no_grad()
    def _val_eval(self, panel, eligible, crit):
        self.model.eval()
        L, H = self.seq, self.H; a = self.train_target_end - self.seq; bs = 1024
        loss_tot, n_tot = 0.0, 0; preds, ys = [], []
        for lo in range(0, len(eligible), bs):
            sl = eligible[lo:lo + bs]; a_arr = np.full(len(sl), a, dtype=np.int64)
            p, f, y, st = _extract_windows(panel, sl, a_arr, L, H, self.p_mean, self.p_scale, self.f_mean, self.f_scale, self.t_mean, self.t_scale, self.log_target)
            pt = torch.tensor(p, dtype=torch.float32, device=self.device); ft = torch.tensor(f, dtype=torch.float32, device=self.device)
            sd = {c: torch.tensor(st[:, j], dtype=torch.long, device=self.device) for j, c in enumerate(self.sc)}
            yt = torch.tensor(y, dtype=torch.float32, device=self.device)
            q, _ = self.model(pt, ft, sd)
            
            # SỬ DỤNG MSE LOSS
            loss_tot += float(F.mse_loss(q[:, :, 1], yt).item()) * len(sl)
            n_tot += len(sl)
            
            med = q[:, :, 1].detach().cpu().numpy().astype(np.float64)
            v = self.ts.inverse_transform(med.reshape(-1, 1)).reshape(med.shape)
            if self.log_target: v = np.expm1(v)
            preds.append(np.clip(v, 0, None))
            ys.append(panel.past[0, sl, a + L:a + L + H].astype(np.float64))
        loss = loss_tot / max(n_tot, 1)
        P = np.concatenate(preds, axis=0) if preds else np.zeros((0, H)); Y = np.concatenate(ys, axis=0) if ys else np.zeros((0, H))
        if P.size:
            den = np.abs(Y).sum(); wmape = float('nan') if den == 0 else float(np.abs(P - Y).sum() / den * 100)
            mae = float(np.abs(P - Y).mean())
        else: wmape = float('nan'); mae = float('nan')
        self.model.train()
        return loss, n_tot, wmape, mae

    def fit(self, panel, eligible):
        import time
        random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
        eligible = np.asarray(eligible, dtype=np.int64)
        print(f'[TFT] train {len(eligible):,} series on {self.device} | MSE LOSS | targets d_{self.train_target_start}..d_{self.train_target_end}')
        t_start = time.time()
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

        for j, c in enumerate(self.sc):
            names = getattr(panel, 'static_values', {}).get(c)
            if names is not None:
                self.maps[c] = {str(names[k]): k + 1 for k in range(len(names))}; self.cards[c] = len(names)
            else:
                u = sorted({int(v) for v in np.unique(panel.static[eligible, j]) if v > 0})
                self.maps[c] = {str(v): i + 1 for i, v in enumerate(u)}; self.cards[c] = len(u)

        self._fit_scalers(panel, eligible)
        ds = MatrixTFTDataset(panel, eligible, seq=self.seq, H=self.H, target_start=self.train_target_start, target_end=self.train_target_end,
                               p_mean=self.p_mean, p_scale=self.p_scale, f_mean=self.f_mean, f_scale=self.f_scale,
                               t_mean=self.t_mean, t_scale=self.t_scale, log_target=self.log_target, **self.sampling)
        self.n_windows = len(ds)
        loader = DataLoader(ds, batch_size=self.batch, shuffle=True, num_workers=0, collate_fn=_make_collate(panel, self.seq, self.H, self.p_mean, self.p_scale, self.f_mean, self.f_scale, self.t_mean, self.t_scale, self.log_target), pin_memory=(self.device == 'cuda'))
        
        self.model = TFTGlobalNet(len(self.pc), len(self.fc), self.cards, self.d, self.heads, drop=self.dropout, seq=self.seq, H=self.H).to(self.device)
        # SỬ DỤNG MSE LOSS
        crit = nn.MSELoss() 
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=1e-5)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=self.lr, total_steps=max(1, (len(ds) // self.batch + 1) * self.epochs), pct_start=0.3)
        scaler_amp, _ = _new_amp() if self.device == 'cuda' else (None, True)

        best, best_state, bad = float('inf'), None, 0
        for ep in range(1, self.epochs + 1):
            self.model.train(); tot = 0.0
            for px, fx, sx, y in loader:
                px, fx, y = px.to(self.device), fx.to(self.device), y.to(self.device)
                sd = {c: sx[:, i].to(self.device) for i, c in enumerate(self.sc)}
                opt.zero_grad()
                if self.device == 'cuda':
                    with _autocast():
                        qp, _ = self.model(px, fx, sd)
                        # Lấy output thứ 1 làm Mean, bỏ qua 0 và 2
                        loss = crit(qp[:, :, 1], y) 
                    scaler_amp.scale(loss).backward(); scaler_amp.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    scaler_amp.step(opt); scaler_amp.update()
                else:
                    qp, _ = self.model(px, fx, sd)
                    loss = crit(qp[:, :, 1], y)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0); opt.step()
                sched.step()
                tot += loss.item() * len(px)
            tot /= max(1, len(ds))
            vl, n_val, vw, vm = self._val_eval(panel, eligible, crit)
            lr_now = float(opt.param_groups[0]['lr'])
            print(f'[TFT] ep {ep}/{self.epochs} train={tot:.5f} val={vl:.5f} val_WMAPE={vw:.2f}% val_MAE={vm:.3f} lr={lr_now:.2e}', flush=True)
            self.history.append({'epoch': ep, 'train_loss': float(tot), 'val_loss': float(vl), 'val_wmape': float(vw), 'val_mae': float(vm), 'lr': lr_now})
            if vl < best - 1e-6:
                best = vl; best_state = copy.deepcopy(self.model.state_dict()); bad = 0
                self.best_epoch = ep; self.best_val_loss = float(vl)
            else: bad += 1
            if bad >= self.patience: break
        if best_state is not None: self.model.load_state_dict(best_state, strict=True)
        self.val_series = int(n_val); self.train_duration_s = float(time.time() - t_start)
        return self

    @torch.no_grad()
    def predict_panel(self, panel, sidx, a):
        self.model.eval()
        L, H = self.seq, self.H; sidx = np.asarray(sidx, dtype=np.int64)
        a_arr = np.full(len(sidx), int(a), dtype=np.int64)
        p, f, _, st = _extract_windows(panel, sidx, a_arr, L, H, self.p_mean, self.p_scale, self.f_mean, self.f_scale, self.t_mean, self.t_scale, self.log_target)
        pt = torch.tensor(p, dtype=torch.float32, device=self.device); ft = torch.tensor(f, dtype=torch.float32, device=self.device)
        sd = {c: torch.tensor(st[:, j], dtype=torch.long, device=self.device) for j, c in enumerate(self.sc)}
        q, _ = self.model(pt, ft, sd)
        
        # SỬ DỤNG OUTPUT THỨ 1 (MEAN)
        med = q[:, :, 1].detach().cpu().numpy().astype(np.float64)
        v = self.ts.inverse_transform(med.reshape(-1, 1)).reshape(med.shape)
        if self.log_target: v = np.expm1(v)
        return np.clip(v, 0, None)

    def save(self, path):
        import pickle
        torch.save(self.model.state_dict(), path + '.weights.pt')
        meta = {'maps': self.maps, 'cards': self.cards, 'ps': self.ps, 'fs': self.fs, 'ts': self.ts, 'pc': self.pc, 'fc': self.fc, 'sc': self.sc, 'target': self.target, 'seq': self.seq, 'H': self.H, 'd': self.d, 'heads': self.heads, 'log_target': self.log_target, 'dropout': self.dropout, 'patience': self.patience, 'sampling': self.sampling, 'lr': self.lr, 'batch': self.batch, 'epochs': self.epochs, 'history': self.history, 'best_epoch': self.best_epoch, 'best_val_loss': self.best_val_loss, 'stop_reason': self.stop_reason, 'boundaries': {'train_target_start': self.train_target_start, 'train_target_end': self.train_target_end, 'val_start': self.val_start, 'val_end': self.val_end, 'test_start': int(globals().get('TEST_START', self.val_end + 1)), 'test_end': int(globals().get('TEST_END', self.val_end + self.H))}}
        with open(path + '.meta.pkl', 'wb') as f: pickle.dump(meta, f)

print('TFT model (MSE Loss) OK')

TFT model (MSE Loss) OK


In [7]:
import gc, warnings, json
warnings.filterwarnings('ignore')

TRAIN_SAMPLE_N = None
PAST = list(PANEL.past_names); FUT = list(PANEL.fut_names); STATIC = list(PANEL.static_names)
_train_end_idx = TRAIN_TARGET_END - 1
_a_max = TRAIN_TARGET_END - SEQ - H
_start = PANEL.start_idx

eligible = []
for _s in range(PANEL.n_series):
    _d0 = int(_start[_s])
    if _d0 < 0 or (_train_end_idx - _d0 + 1) < SEQ or _d0 > _a_max: continue
    eligible.append(_s)
eligible = np.asarray(eligible, dtype=np.int64)

tft = TFTGlobalForecaster(PAST, FUT, STATIC, target='sales', seq=SEQ, H=H, d=TFT_CFG['d_model'], heads=TFT_CFG['num_heads'], lr=TFT_CFG['lr'], batch=TFT_CFG['batch_size'], epochs=TFT_CFG['epochs'], log_target=LOG_TARGET, dropout=TFT_CFG.get('dropout', 0.1), patience=TFT_CFG.get('patience', 5), sampling=SAMPLING).fit(PANEL, eligible)

tft.save(os.path.join(OUT_DIR, 'tft_global_full30490'))
print('Checkpoint saved.')
del eligible; gc.collect()

[TFT] train 30,466 series on cpu | MSE LOSS | targets d_57..d_1857
[scaler] past cells=44,319,566 | target mean=0.4927 scale=0.7247 | fut weight=44,319,566


KeyboardInterrupt: 

In [ ]:
import gc, glob, hashlib, json, os, pickle, shutil
import numpy as np, pandas as pd, torch

_EXPECTED_BOUNDARIES = {'train_target_start': int(TRAIN_TARGET_START), 'train_target_end': int(TRAIN_TARGET_END), 'val_start': int(VAL_START), 'val_end': int(VAL_END), 'test_start': int(TEST_START), 'test_end': int(TEST_END)}
if 'tft' not in globals():
    _w = os.path.join(OUT_DIR, 'tft_global_full30490.weights.pt')
    _m = os.path.join(OUT_DIR, 'tft_global_full30490.meta.pkl')
    with open(_m, 'rb') as f: _meta = pickle.load(f)
    tft = TFTGlobalForecaster(_meta['pc'], _meta['fc'], _meta['sc'], target=_meta['target'], seq=_meta['seq'], H=_meta['H'], d=_meta['d'], heads=_meta['heads'], log_target=_meta['log_target'])
    tft.maps, tft.cards = _meta['maps'], _meta['cards']
    tft.ps, tft.fs, tft.ts = _meta['ps'], _meta['fs'], _meta['ts']
    tft.model = TFTGlobalNet(len(tft.pc), len(tft.fc), tft.cards, tft.d, tft.heads, drop=tft.dropout, seq=tft.seq, H=tft.H).to(tft.device)
    tft.model.load_state_dict(torch.load(_w, map_location='cpu'), strict=True)
    tft.model.eval()
    tft._sync_scalers()

SERIES_IDS = PANEL.ids
N_SERIES = PANEL.n_series
SPLIT_SPECS = {
    'val': {'origin': int(TRAIN_TARGET_END), 'a': int(TRAIN_TARGET_END - SEQ)},
    'test': {'origin': int(VAL_END), 'a': int(VAL_END - SEQ)}
}
PRED_RESULTS = {}
for split in ['val', 'test']:
    spec = SPLIT_SPECS[split]
    a = spec['a']
    pred = np.zeros((N_SERIES, H), dtype=np.float64)
    true = np.zeros((N_SERIES, H), dtype=np.float64)
    done = np.zeros(N_SERIES, dtype=bool)
    has_ctx = (PANEL.start_idx >= 0) & (PANEL.start_idx <= a)
    todo = np.where(has_ctx)[0]
    BS = 512
    for lo in range(0, len(todo), BS):
        sidx = todo[lo:lo + BS]
        p = tft.predict_panel(PANEL, sidx, a)
        y = PANEL.past[0, sidx, a + SEQ:a + SEQ + H].astype(np.float64)
        pred[sidx] = p
        true[sidx] = y
        done[sidx] = True
    PRED_RESULTS[split] = {'pred': pred, 'true': true, 'done': done, 'spec': spec}
print('PREDICT DONE')

In [ ]:
import os, numpy as np, pandas as pd

SCORE_LEVELS = ['L12', 'L9', 'L3', 'L2', 'L1']
REPORT_LEVELS = ['L12', 'L10', 'L9', 'L3', 'L2', 'L1']
BASE_LEVEL = 'L12'
ALL_LEVELS = REPORT_LEVELS

def _agg(G, mat): return np.asarray(G.dot(mat), dtype=np.float64)

def _level_metrics(y, p, y_train_mat=None):
    n_keys, h = y.shape
    ms = []
    for i in range(n_keys):
        tr = y_train_mat[i] if y_train_mat is not None else None
        ms.append(forecast_metrics(y[i], p[i], y_train=tr))
    out = {}
    for k in ('RMSE', 'MAE', 'sMAPE (%)', 'Bias'): out[k] = float(np.mean([m[k] for m in ms]))
    mapes = np.asarray([m['MAPE (%)'] for m in ms], dtype=np.float64)
    out['n_zero_mape_keys'] = int(np.isnan(mapes).sum())
    out['MAPE (%)'] = float(np.nanmean(mapes)) if out['n_zero_mape_keys'] < len(mapes) else float('nan')
    wmapes = np.asarray([m['WMAPE (%)'] for m in ms], dtype=np.float64)
    out['n_zero_keys'] = int(np.isnan(wmapes).sum())
    out['WMAPE (%)'] = float(np.nanmean(wmapes)) if out['n_zero_keys'] < len(wmapes) else float('nan')
    if y_train_mat is not None:
        rmsses = np.asarray([m['RMSSE'] for m in ms], dtype=np.float64)
        out['RMSSE'] = float(np.nanmean(rmsses)) if np.any(~np.isnan(rmsses)) else float('nan')
    else: out['RMSSE'] = float('nan')
    y64, p64 = y.reshape(-1), np.clip(p.reshape(-1), 0, None)
    out['pooled_WMAPE (%)'] = calc_wmape(y64, p64)
    out['pooled_MAPE (%)'] = calc_mape(y64, p64)
    out['total_actual'] = float(y64.sum()); out['total_pred'] = float(p64.sum())
    out['pred_actual_ratio'] = float(p64.sum() / y64.sum()) if y64.sum() != 0 else float('nan')
    out['zero_true_keys'] = int((y.sum(axis=1) == 0).sum())
    out['n_keys'] = int(n_keys)
    return out

RESULTS = []
for split, R in PRED_RESULTS.items():
    done = R['done']; a = R['spec']['a']; origin = R['spec']['origin']
    pred = R['pred'][done]; true = R['true'][done]
    nlo, nhi = origin - H, origin - 1
    naive28 = PANEL.past[0][:, nlo:nhi + 1].astype(np.float64)[done]
    rows_done = np.where(done)[0]
    
    train_sales_base = PANEL.past[0, rows_done, :origin].astype(np.float64)
    train_prices_base = PANEL.prices[rows_done, :origin].astype(np.float64)
    last_28_sales = train_sales_base[:, -28:]; last_28_prices = train_prices_base[:, -28:]
    row_medians = np.nan_to_num(np.nanmedian(train_prices_base, axis=1, keepdims=True), nan=1.0)
    last_28_prices = np.where(np.isnan(last_28_prices), row_medians, last_28_prices)
    dollar_sales_28_base = np.sum(last_28_sales * last_28_prices, axis=1)

    for model_name, model_pred in [('TFT', pred), ('naive28', naive28)]:
        level_wrmsse = {}
        for lv in REPORT_LEVELS:
            if lv == BASE_LEVEL:
                yk, pk, tr_k, ds_k = true, model_pred, train_sales_base, dollar_sales_28_base
            else:
                G = PANEL.groups[lv][1][:, rows_done]
                yk, pk = _agg(G, true), _agg(G, model_pred)
                tr_k = _agg(G, train_sales_base)
                ds_k = np.asarray(G.dot(dollar_sales_28_base), dtype=np.float64)
            
            mt = _level_metrics(yk, pk, y_train_mat=tr_k)
            w_k = ds_k / ds_k.sum() if ds_k.sum() > 0 else np.full_like(ds_k, 1.0 / len(ds_k))
            r_t = np.asarray([calc_rmsse(yk[i], pk[i], tr_k[i]) for i in range(yk.shape[0])], dtype=np.float64)
            valid_t = ~np.isnan(r_t)
            norm_w = w_k[valid_t] / w_k[valid_t].sum() if w_k[valid_t].sum() > 0 else w_k[valid_t]
            level_wrmsse[lv] = float(np.sum(r_t[valid_t] * norm_w))

            if lv in SCORE_LEVELS:
                RESULTS.append({
                    'split': split, 'level': lv, 'model': model_name,
                    'WMAPE (%)': mt['WMAPE (%)'], 'MAPE (%)': mt['MAPE (%)'],
                    'RMSSE': mt['RMSSE'], 'WRMSSE': level_wrmsse[lv],
                    'MAE': mt['MAE'], 'RMSE': mt['RMSE'], 'Bias': mt['Bias'],
                    'pred_actual_ratio': mt['pred_actual_ratio'],
                    'n_zero_keys': mt['n_zero_keys'], 'n_zero_mape_keys': mt['n_zero_mape_keys']
                })
        
        if model_name == 'TFT':
            wrmsse_overall = float(np.mean([level_wrmsse[lv] for lv in SCORE_LEVELS]))
            print(f"[{split}] TFT WRMSSE5 = {wrmsse_overall:.4f}")

EVAL_RESULTS = pd.DataFrame(RESULTS)
print('\n' + '=' * 120)
print('Phase 0 EVAL (MSE Loss)')
print('=' * 120)
print(EVAL_RESULTS.to_string(index=False, float_format=lambda x: f'{x:,.4f}'))
EVAL_RESULTS.to_csv(os.path.join(OUT_DIR, 'phase0_eval.csv'), index=False)

In [ ]:
import hashlib, json, numpy as np, pandas as pd, torch

_PASS, _FAIL = 0, []
def _check(name, cond, detail=''):
    global _PASS
    ok = bool(cond)
    if ok: _PASS += 1
    else: _FAIL.append((name, str(detail)))
    print(('[PASS] ' if ok else '[FAIL] ') + name + ('' if ok else f' :: {detail}'))

_check('calc_mape known value', abs(calc_mape([0, 2, 4], [1, 3, 5]) - 37.5) < 1e-6)
_check('calc_rmsse known value', abs(calc_rmsse([2, 4], [4, 6], [0, 0, 2, 4]) - 1.0) < 1e-6)
_check('eval metrics finite', bool(np.isfinite(EVAL_RESULTS[['WMAPE (%)', 'MAPE (%)', 'RMSSE', 'WRMSSE']].to_numpy(dtype=np.float64)).all()))

print('\n' + '=' * 60)
print(f'AUDIT SUMMARY: {_PASS} passed, {len(_FAIL)} failed')
if _FAIL:
    for _n, _d in _FAIL: print('  FAILED:', _n, '::', _d)
    raise AssertionError(f'{len(_FAIL)} audit checks failed.')
print('ALL AUDIT CHECKS PASSED')